# Klasifikasi DemogPairs Menggunakan ViT (Wajah dan Umur) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx,subject_id
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5,able_wanamakok
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5,able_wanamakok
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5,able_wanamakok
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5,able_wanamakok
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5,able_wanamakok
...,...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3,zachary_quinto
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3,zachary_quinto
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3,zachary_quinto
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3,zachary_quinto


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

 10%|▉         | 1026/10800 [00:00<00:00, 10251.53it/s]

 19%|█▉        | 2095/10800 [00:00<00:00, 10505.17it/s]

 29%|██▉       | 3146/10800 [00:00<00:00, 10367.06it/s]

 40%|███▉      | 4301/10800 [00:00<00:00, 10828.60it/s]

...

 71%|███████   | 7658/10800 [00:00<00:00, 11055.19it/s]

 81%|████████  | 8764/10800 [00:00<00:00, 10953.92it/s]

 91%|█████████▏| 9860/10800 [00:00<00:00, 10941.30it/s]

100%|██████████| 10800/10800 [00:01<00:00, 10790.85it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
train_idx, test_idx = u.split_demogpairs_subject_disjoint(data, train_ratio=0.8, random_state=42)

X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
groups = np.array([d['subject_id'] for d in data])

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

train_subs = set(groups_train)
test_subs = set(groups_test)
overlap = len(train_subs.intersection(test_subs))

print("=== VERIFIKASI PARTISI SUBJECT-DISJOINT ===")
print(f"Train samples: {len(X_train)} ({len(train_subs)} subjek unik)")
print(f"Test samples : {len(X_test)} ({len(test_subs)} subjek unik)")
print(f"Tumpang tindih identitas (Identity Overlap): {overlap} subjek (0% Leakage)")
print("\nDistribusi kelas Train Set:")
for c_name in u.demogpairs_classes:
    c_idx = u.demogpairs_label_to_idx[c_name]
    print(f" - {c_name:<15}: {(y_train == c_idx).sum()} citra ({(y_train == c_idx).sum() // 18} subjek)")
print("\nDistribusi kelas Test Set:")
for c_name in u.demogpairs_classes:
    c_idx = u.demogpairs_label_to_idx[c_name]
    print(f" - {c_name:<15}: {(y_test == c_idx).sum()} citra ({(y_test == c_idx).sum() // 18} subjek)")

=== VERIFIKASI PARTISI SUBJECT-DISJOINT ===
Train samples: 8640 (480 subjek unik)
Test samples : 2160 (120 subjek unik)
Tumpang tindih identitas (Identity Overlap): 0 subjek (0% Leakage)

Distribusi kelas Train Set:
 - Asian_Females  : 1440 citra (80 subjek)
 - Asian_Males    : 1440 citra (80 subjek)
 - Black_Females  : 1440 citra (80 subjek)
 - Black_Males    : 1440 citra (80 subjek)
 - White_Females  : 1440 citra (80 subjek)
 - White_Males    : 1440 citra (80 subjek)

Distribusi kelas Test Set:
 - Asian_Females  : 360 citra (20 subjek)
 - Asian_Males    : 360 citra (20 subjek)
 - Black_Females  : 360 citra (20 subjek)
 - Black_Males    : 360 citra (20 subjek)
 - White_Females  : 360 citra (20 subjek)
 - White_Males    : 360 citra (20 subjek)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-face-age_",
    results_path="results/demogpairs_lr_vit-face-age_",
    groups_train=groups_train,
    groups_test=groups_test
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8740740740740741
Precision : 0.8743599298140098
Recall    : 0.8740740740740739
F1 Score  : 0.8739866010900704
Inference Latency : 0.0056 ms/sample (0.0120s total for 2160 samples)
Model Storage Size: 0.11 MB
               precision    recall  f1-score   support

Asian_Females     0.8540    0.8611    0.8575       360
  Asian_Males     0.8211    0.8667    0.8432       360
Black_Females     0.8496    0.8000    0.8240       360
  Black_Males     0.8799    0.8750    0.8774       360
White_Females     0.9000    0.9000    0.9000       360
  White_Males     0.9417    0.9417    0.9417       360

     accuracy                         0.8741      2160
    macro avg     0.8744    0.8741    0.8740      2160
 weighted avg     0.8744    0.8741    0.8740      2160

Recall    : 0.8740740740740739
F1 Score  : 0.8739866010900704
               precision    recall  f1-score   support

Asian_Females     0.8540    0.8611    0.8575       360
  Asian_Males     0.8211    0.8667    0.8432       3

Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9523148148148148,0.8539944903581267,0.8611111111111112,0.8575380359612725,360
Asian_Males,0.9462962962962963,0.8210526315789474,0.8666666666666667,0.8432432432432433,360
Black_Females,0.9430555555555555,0.8495575221238938,0.8,0.8240343347639485,360
Black_Males,0.9592592592592593,0.8798882681564246,0.875,0.8774373259052926,360
White_Females,0.9666666666666667,0.9,0.9,0.9,360
White_Males,0.9805555555555555,0.9416666666666667,0.9416666666666667,0.9416666666666667,360


Confusion matrix saved: images\cm_lr_vit-face-age_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               310                17                24                 0                 8                 1
         Asian_Males                17               312                 3                23                 0                 5
       Black_Females                17                14               288                13                26                 2
         Black_Males                 0                22                13               315                 0                10
       White_Females                19                 3                11                 0               324                 3
         White_Males                 0                12                 0                 7                 2               339


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,train_time_seconds,inference_time_seconds,latency_ms_per_sample,model_size_mb,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-face-age_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8740740740740741,0.8739866010900704,0.8743599298140098,0.8740740740740739,1825.3294146060944,0.011994500004220754,0.0055530092612133125,0.11349010467529297,96


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-face-age_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 1825.0,
 'days': 0,
 'hours': 0,
 'minutes': 30,
 'seconds': 25.0,
 'text': '0 hari 0 jam 30 menit 25.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 20558.0,
 'days': 0,
 'hours': 5,
 'minutes': 42,
 'seconds': 38.0,
 'text': '0 hari 5 jam 42 menit 38.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.89,0.8524,0.8924,0.9242,0.897,0.8912,0.8909,0.8921,0.8912,52.2849
2,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.89,0.8524,0.8924,0.9242,0.897,0.8912,0.8909,0.8921,0.8912,51.3878
3,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8895,0.8513,0.8935,0.9242,0.897,0.8911,0.8908,0.892,0.8911,70.8804
4,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8895,0.8513,0.8935,0.9242,0.897,0.8911,0.8908,0.892,0.8911,72.7477
...,...,...,...,...,...,...,...,...,...,...,...
93,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8611,0.8212,0.849,0.8941,0.86,0.8571,0.856,0.8578,0.8571,15.3963
94,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8611,0.8212,0.849,0.8941,0.86,0.8571,0.856,0.8578,0.8571,10.3462
95,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8611,0.8212,0.849,0.8941,0.86,0.8571,0.856,0.8578,0.8571,16.0
96,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8611,0.8212,0.849,0.8941,0.86,0.8571,0.856,0.8578,0.8571,8.1464
